# Classify Raisins with Feature Importance + Hyperparameter Tuning
## Practice Skeleton

**Short name (GitHub):** `HypeTune_Py`

**Lab sources:** Codecademy *Classify Raisins with Hyperparameter Tuning* + *Feature Importance* + *Hyperparameter Tuning with scikit-learn* (GridSearchCV / RandomizedSearchCV). Dataset: 900 raisins × 7 morphometrics, balanced Class 0/1 (Kecimen / Besni).

**How to use this notebook**
- Work top-to-bottom. Cells marked `# YOUR CODE HERE` are the practice.
- Keep the inline cheat-sheet and flowchart cells visible.
- Compare with `HypeTune_Py_Solution.ipynb` only after you have a working attempt.
- Charts live next to this file (`hypetune_*.png`). Data: `data/Raisin_Dataset.csv`.
- Re-use the same pipeline on a new table with `HypeTune_Py_Reusable_Template.ipynb`.


## Inline cheat-sheet (keep this cell visible)

See also **`HypeTune_Py_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Parameter vs hyperparameter | Coefficients / split thresholds are **learned**. `max_depth`, `C`, `penalty`, `k` are **chosen**. |
| Gini importance | `clf.feature_importances_` after a tree/forest fit with `criterion='gini'`. Biased toward high-cardinality numeric features; ignores correlation. |
| Permutation importance | Shuffle one column, measure drop in score. Model-agnostic, needs a held-out set, slower. `sklearn.inspection.permutation_importance`. |
| Scaled \|coef\| | Only comparable after `StandardScaler`. L1 can zero features. |
| Grid search | Exhaustive Cartesian product of *lists*. `GridSearchCV(est, param_grid, cv=5)`. |
| Random search | Sample `n_iter` draws from *distributions*. `RandomizedSearchCV(est, param_distributions, n_iter=8)`. |
| `uniform(loc, scale)` | Draws on `[loc, loc+scale]`. `uniform(0, 100)` → C ∈ [0, 100]. |
| Attributes after `.fit` | `.best_estimator_`, `.best_params_`, `.best_score_` (mean CV), `.cv_results_`, `.score(X_test, y_test)`. |
| Never | Report `.best_score_` as the final generalisation number. That fold was used to *pick* the hyperparams. |
| `liblinear` | Needed if you still pass `penalty='l1'` on `LogisticRegression`. sklearn ≥ 1.8 prefers `l1_ratio` (0 = L2, 1 = L1). |
| Split once | Freeze `random_state`. Do not retune on the test fold. |


## Flowchart of the desired outcome

![HypeTune flow](hypetune_flowchart.png)

Inspect the balanced 900-row card → estimate which morphometrics actually move the class → exhaust a small tree grid → sample a continuous `C` for logistic regression → confirm on the hold-out fold → poke the knobs in the simulation cell.


## 0. Packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.stats import uniform, loguniform

np.random.seed(19)
plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 1. Load and inspect the raisins table

900 rows, 7 morphometrics (`Area`, `MajorAxisLength`, `MinorAxisLength`, `Eccentricity`, `ConvexArea`, `Extent`, `Perimeter`) plus binary `Class`. Balanced 450 / 450. No missing cells.

![class snapshot](hypetune_data.png)

In [ ]:
# YOUR CODE HERE
df = pd.read_csv("data/Raisin_Dataset.csv")
print(df.head())
# shape, columns, Class value_counts, missing count, describe


## 2. Predictor matrix `X` and target `y`

In [ ]:
# YOUR CODE HERE
X = None
y = None
# print n features, n samples, class-1 count


## 3. Train / test split

Codecademy uses `train_test_split(X, y, random_state=19)` with the default 75/25. Freeze that seed for the rest of the notebook.

In [ ]:
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None


## 4. Feature importance — Gini on one tree

Gini impurity at a node is $1 - \sum_k p_k^2$. Gini *gain* is the drop after a split. sklearn stores the total gain attributed to each feature in `feature_importances_`, normalised to sum to 1.

**Watch-outs:** high-cardinality numeric features win ties; correlated size features steal credit from each other.

In [ ]:
# YOUR CODE HERE
# Fit DecisionTreeClassifier(criterion='gini') on the training fold.
# Print feature_importances_ aligned with X.columns and draw a barh chart.
dt = None
gini_dt = None


## 5. Aggregate Gini (random forest) and permutation importance

A forest averages Gini across trees that each saw a feature subsample. Permutation importance shuffles one column on the *test* fold and records the drop in accuracy — model-agnostic, correlation-aware in a different way (a redundant feature can look useless once its twin is present).

![importance panel](hypetune_importance.png)

In [ ]:
# YOUR CODE HERE
# Fit RandomForestClassifier(n_estimators=200, random_state=19).
# Print forest feature_importances_ and permutation_importance on X_test.
rf = None


## 6. Grid search — Decision Tree

### 6a. Estimator

Lesson constructor: `DecisionTreeClassifier()` with defaults. Depth and split minimum stay as hyperparameters we are about to search.

In [ ]:
# YOUR CODE HERE
tree = None


### 6b. `param_grid`

Two lists → 3 × 3 = 9 combinations. Grid search tries every pair.

In [ ]:
# YOUR CODE HERE
parameters = None


### 6c. `GridSearchCV` + fit

Default `cv=5` so every training row is in a validation fold once. That is 9 × 5 = 45 tree fits, then one refit of the winner on the full training fold (`refit=True`).

In [ ]:
# YOUR CODE HERE
grid = None
# fit on X_train, y_train


### 6d. Read the winner

`.best_estimator_` is the refitted model. `.best_score_` is the mean CV accuracy of that setting — **not** the number you quote as final performance.

In [ ]:
# YOUR CODE HERE
# print best_estimator_, best_params_, best_score_, and grid.score(X_test, y_test)


### 6e. The 9-cell table

![grid heatmap](hypetune_grid.png)

Depth 5 beats 3 (underfit) and 7 (starts to overfit). `min_samples_split` barely moves the needle on this card.

In [ ]:
# YOUR CODE HERE
# Concatenate cv_results_['params'] with mean_test_score as a 'Score' column and print it.


## 7. Random search — Logistic Regression

### 7a. Estimator

`solver='liblinear'` is the lesson choice because it supports both L1 and L2. `max_iter=1000` keeps the lbfgs-style convergence warning away.

In [ ]:
# YOUR CODE HERE
lr = None


### 7b. Distributions

A Python list is treated as a discrete uniform. `uniform(loc=0, scale=100)` is continuous on [0, 100]. Draw a few values with `.rvs()` to confirm the range before you search.

In [ ]:
# YOUR CODE HERE
# penalty: ['l1','l2']   C: uniform(loc=0, scale=100)
distributions = None


### 7c. `RandomizedSearchCV` + fit

`n_iter=8` means eight random `(penalty, C)` pairs, each with 5-fold CV → 40 logistic fits. Pass `random_state` so the draws do not jump between reruns.

In [ ]:
# YOUR CODE HERE
clf = None
# fit on the training fold


### 7d. Winner + table

![random search](hypetune_random.png)

On unscaled raisins the likelihood surface is flat across a wide band of C. Expect several pairs to land within ~0.3 pp of each other.

In [ ]:
# YOUR CODE HERE
# print best_estimator_, best_score_, and a table of params + mean_test_score sorted by Accuracy


## 8. Alternate code (same question, different route)

Four replacements you can drop in on a new project: locked seed on the tree, log-uniform C, a scaled pipeline so coefficients are feature importances, and a raw double loop so GridSearchCV is not magic.

In [ ]:
# YOUR CODE HERE — pick at least two
# A. GridSearchCV with DecisionTreeClassifier(random_state=19)
# B. RandomizedSearchCV with C ~ loguniform(1e-2, 1e2) and n_iter=16
# C. Pipeline(StandardScaler, LogisticRegression) + GridSearchCV over lr__C and lr__penalty
# D. Nested for-loop over max_depth × min_samples_split (no GridSearchCV)


## 9. More practice

Three short drills that reuse the same split: a third tree hyperparameter, a collinearity drop, and the breast-cancer card from the lesson notebooks.

In [ ]:
# YOUR CODE HERE — three short drills
# 1. Add min_samples_leaf: [1, 5, 10] to the tree grid and refit.
# 2. Drop Area and ConvexArea (collinear size features), re-run the original 3×3 tree grid.
# 3. load_breast_cancer() + GridSearchCV on LogisticRegression the way the lesson notebooks did
#    (penalty in {l1,l2}, C in {1,10,100}, solver='liblinear').


## 10. Simulation — change a few knobs

![simulation](hypetune_simulation.png)

Edit `N_ITER`, `TRAIN_N`, `FLIP_P`, `N_JUNK` and re-run. The three sweep charts stay as a map of the neighbourhood around your current setting.

In [ ]:
# --- knobs (edit these and re-run) ---
N_ITER = 8
TRAIN_N = 675
FLIP_P = 0.00
N_JUNK = 0
RS = 19

# YOUR CODE HERE
# 1. Slice the training fold to TRAIN_N rows.
# 2. Optionally append N_JUNK Gaussian columns to train and test.
# 3. Optionally flip FLIP_P of the training labels.
# 4. Refit the 3×3 DecisionTree grid and the n_iter Logistic random search.
# 5. Print best params + CV + test for both, then draw the three sweeps
#    (n_iter, train n, flip %) with a red line on the current knob.


## 11. Audience rewrite

Use the two attached PDFs (*What to Consider When Considering the Audience*, *Audience and Situation Analysis*). Four cuts of the same result: Experts / Technicians / Executives / Nonspecialists.

In [ ]:
# YOUR CODE HERE
# Rewrite the same findings four ways, using the attached audience PDFs:
#   Experts · Technicians · Executives · Nonspecialists
# Cover: which feature actually matters, why CV ≠ test, what a manager should do.


## 12. Takeaways

In [ ]:
# Write five to seven bullets after you finish the core tasks, the alternates, more practice, and the simulation.
